# submission first cf v1

- score submissions first
- aggregate to collaborations
- serve collaboration recommendations


In [1]:
from __future__ import annotations

from ast import literal_eval
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
DATA_DIR = ROOT / "data" / "synthetic"


## Load


In [2]:
collab_table = pd.read_csv(DATA_DIR / "collaboration_training_table.csv")
submission_table = pd.read_csv(DATA_DIR / "submission_training_table.csv")
collabs = pd.read_csv(DATA_DIR / "collaborations.csv")
subs = pd.read_csv(DATA_DIR / "submissions.csv")

collab_table["lastEventAt"] = pd.to_datetime(collab_table["lastEventAt"], utc=True)
submission_table["lastEventAt"] = pd.to_datetime(submission_table["lastEventAt"], utc=True)
collab_table["tagsKey"] = collab_table["tagsKey"].apply(literal_eval)
collabs["tagsKey"] = collabs["tagsKey"].apply(literal_eval)

active_collabs = collabs[collabs["status"].isin(["submission", "voting"])].copy()
active_collabs = active_collabs[["id", "projectId", "name", "status", "tagsKey", "publishedAt"]].rename(columns={"id": "collaborationId"})

subs = subs[["trackPath", "projectId", "collaborationId"]]
submission_table.head()

,userId,projectId,collaborationId,trackPath,submission_favorite,submission_like,submission_vote,totalEventWeight,lastEventAt
0,user-0001,project-007,collab-007-04,collaborations/collab-007-04/submissions/sub-0...,1,1,0,3.0,2026-04-24 00:22:43+00:00
1,user-0001,project-003,collab-003-01,collaborations/collab-003-01/submissions/sub-0...,1,1,0,3.0,2026-04-03 07:12:35+00:00
2,user-0001,project-009,collab-009-02,collaborations/collab-009-02/submissions/sub-0...,0,1,0,1.0,2026-03-28 11:54:51+00:00
3,user-0001,project-009,collab-009-02,collaborations/collab-009-02/submissions/sub-0...,0,1,0,1.0,2026-03-19 12:13:59+00:00
4,user-0001,project-003,collab-003-01,collaborations/collab-003-01/submissions/sub-0...,0,1,0,1.0,2026-01-14 03:42:11+00:00


## Holdout split


In [3]:
eligible_history = collab_table[collab_table["status"].isin(["submission", "voting"])].copy()
eligible_history = eligible_history.sort_values(["userId", "lastEventAt"])

holdout = eligible_history.groupby("userId").tail(1).copy()
train_collab_history = eligible_history.merge(
    holdout[["userId", "collaborationId"]].assign(_holdout=1),
    on=["userId", "collaborationId"],
    how="left",
)
train_collab_history = train_collab_history[train_collab_history["_holdout"].isna()].drop(columns="_holdout")

valid_users = train_collab_history["userId"].value_counts()
valid_users = set(valid_users[valid_users > 0].index) & set(holdout["userId"])
train_collab_history = train_collab_history[train_collab_history["userId"].isin(valid_users)].copy()
holdout = holdout[holdout["userId"].isin(valid_users)].copy()

heldout_collabs = set(holdout["collaborationId"])
train_submission_history = submission_table[submission_table["userId"].isin(valid_users)].copy()
train_submission_history = train_submission_history[~train_submission_history["collaborationId"].isin(heldout_collabs)].copy()

len(valid_users), train_collab_history.shape, train_submission_history.shape, holdout.shape

(239, (1434, 14), (710, 9), (239, 14))

## Submission matrix


In [4]:
submission_interactions = (
    train_submission_history.groupby(["userId", "trackPath"], as_index=False)["totalEventWeight"]
    .sum()
)

submission_matrix = submission_interactions.pivot(index="userId", columns="trackPath", values="totalEventWeight").fillna(0.0)
submission_matrix.shape

(185, 99)

## Submission similarity


In [5]:
X = submission_matrix.to_numpy(dtype=float)
item_matrix = X.T
norms = np.linalg.norm(item_matrix, axis=1, keepdims=True)
norms[norms == 0] = 1.0
item_matrix_norm = item_matrix / norms
similarity = item_matrix_norm @ item_matrix_norm.T
np.fill_diagonal(similarity, 0.0)

item_ids = submission_matrix.columns.tolist()
sim_df = pd.DataFrame(similarity, index=item_ids, columns=item_ids)
sim_df.iloc[:5, :5]

,collaborations/collab-007-01/submissions/sub-007-01-01.mp3,collaborations/collab-007-01/submissions/sub-007-01-03.mp3,collaborations/collab-007-01/submissions/sub-007-01-04.mp3,collaborations/collab-007-01/submissions/sub-007-01-05.mp3,collaborations/collab-007-01/submissions/sub-007-01-06.mp3
collaborations/collab-007-01/submissions/sub-007-01-01.mp3,0.0,0.000000,0.0,0.0,0.000000
collaborations/collab-007-01/submissions/sub-007-01-03.mp3,0.0,0.000000,0.0,0.0,0.179284
collaborations/collab-007-01/submissions/sub-007-01-04.mp3,0.0,0.000000,0.0,0.0,0.000000
collaborations/collab-007-01/submissions/sub-007-01-05.mp3,0.0,0.000000,0.0,0.0,0.000000
collaborations/collab-007-01/submissions/sub-007-01-06.mp3,0.0,0.179284,0.0,0.0,0.000000


## Submission candidate scores


In [6]:
submission_popularity = submission_interactions.groupby("trackPath", as_index=False)["totalEventWeight"].sum()
submission_popularity["submissionPopularityNorm"] = submission_popularity["totalEventWeight"] / (submission_popularity["totalEventWeight"].max() or 1)
submission_pop_map = dict(zip(submission_popularity["trackPath"], submission_popularity["submissionPopularityNorm"]))

seen_submissions = train_submission_history[["userId", "trackPath"]].drop_duplicates().assign(seen=1)
eval_users = pd.DataFrame({"userId": sorted(valid_users)})

candidate_submissions = eval_users.assign(_k=1).merge(subs.assign(_k=1), on="_k").drop(columns="_k")
candidate_submissions = candidate_submissions.merge(active_collabs[["collaborationId"]], on="collaborationId", how="inner")
candidate_submissions = candidate_submissions.merge(seen_submissions, on=["userId", "trackPath"], how="left")
candidate_submissions = candidate_submissions[candidate_submissions["seen"].isna()].drop(columns="seen")

score_rows = []
for user_id, row in submission_matrix.iterrows():
    seen_weights = row[row > 0]
    seen_ids = set(seen_weights.index)
    user_scores = {}
    if len(seen_weights) > 0:
        seen_vector = seen_weights.to_numpy(dtype=float)
        sims = sim_df[seen_weights.index].mul(seen_vector, axis=1).sum(axis=1)
        user_scores = sims.to_dict()
    for track_path in candidate_submissions[candidate_submissions["userId"] == user_id]["trackPath"]:
        if track_path in seen_ids:
            continue
        score_rows.append({
            "userId": user_id,
            "trackPath": track_path,
            "cfScoreRaw": float(user_scores.get(track_path, 0.0)),
            "submissionPopularityNorm": float(submission_pop_map.get(track_path, 0.0)),
        })

submission_scores = pd.DataFrame(score_rows)
max_cf = submission_scores.groupby("userId")["cfScoreRaw"].transform("max").replace(0, 1)
submission_scores["cfScoreNorm"] = submission_scores["cfScoreRaw"] / max_cf
submission_scores["submissionScore"] = 0.85 * submission_scores["cfScoreNorm"] + 0.15 * submission_scores["submissionPopularityNorm"]

submission_scores = submission_scores.merge(subs, on="trackPath", how="left")
submission_scores.head()

,userId,trackPath,cfScoreRaw,submissionPopularityNorm,cfScoreNorm,submissionScore,projectId,collaborationId
0,user-0002,collaborations/collab-001-01/submissions/sub-0...,0.0,0.0,0.0,0.0,project-001,collab-001-01
1,user-0002,collaborations/collab-001-01/submissions/sub-0...,0.0,0.0,0.0,0.0,project-001,collab-001-01
2,user-0002,collaborations/collab-001-01/submissions/sub-0...,0.0,0.0,0.0,0.0,project-001,collab-001-01
3,user-0002,collaborations/collab-001-01/submissions/sub-0...,0.0,0.0,0.0,0.0,project-001,collab-001-01
4,user-0002,collaborations/collab-001-01/submissions/sub-0...,0.0,0.0,0.0,0.0,project-001,collab-001-01


## Aggregate to collaborations


In [7]:
submission_scores = submission_scores.sort_values(["userId", "collaborationId", "submissionScore"], ascending=[True, True, False])

submission_scores["submissionRankInsideCollab"] = submission_scores.groupby(["userId", "collaborationId"]).cumcount() + 1
top_inside = submission_scores[submission_scores["submissionRankInsideCollab"] <= 3].copy()

collab_scores = (
    top_inside.groupby(["userId", "projectId", "collaborationId"], as_index=False)
    .agg(
        aggregatedSubmissionScore=("submissionScore", "mean"),
        bestSubmissionScore=("submissionScore", "max"),
    )
)

best_submission = top_inside.sort_values(["userId", "collaborationId", "submissionScore"], ascending=[True, True, False]).groupby(["userId", "collaborationId"]).head(1)
best_submission = best_submission[["userId", "collaborationId", "trackPath", "submissionScore"]].rename(columns={"trackPath": "highlightedSubmission", "submissionScore": "highlightedSubmissionScore"})

collab_scores = collab_scores.merge(best_submission, on=["userId", "collaborationId"], how="left")
collab_scores = collab_scores.merge(active_collabs, on=["collaborationId", "projectId"], how="left")
collab_scores.head()

,userId,projectId,collaborationId,aggregatedSubmissionScore,bestSubmissionScore,highlightedSubmission,highlightedSubmissionScore,name,status,tagsKey,publishedAt
0,user-0002,project-001,collab-001-01,0.0,0.0,collaborations/collab-001-01/submissions/sub-0...,0.0,Collaboration 1-1,voting,"[acid, breakbeat, cinematic]",2026-04-28T14:26:30Z
1,user-0002,project-001,collab-001-02,0.0,0.0,collaborations/collab-001-02/submissions/sub-0...,0.0,Collaboration 1-2,voting,"[breakbeat, lofi, organic]",2025-12-17T02:06:25Z
2,user-0002,project-002,collab-002-01,0.0,0.0,collaborations/collab-002-01/submissions/sub-0...,0.0,Collaboration 2-1,submission,"[ambient, drum-and-bass]",2026-03-23T22:33:55Z
3,user-0002,project-002,collab-002-02,0.0,0.0,collaborations/collab-002-02/submissions/sub-0...,0.0,Collaboration 2-2,voting,"[idm, uk-bass]",2026-01-16T13:18:16Z
4,user-0002,project-002,collab-002-03,0.0,0.0,collaborations/collab-002-03/submissions/sub-0...,0.0,Collaboration 2-3,submission,[minimal],2026-03-20T00:06:58Z


## Top recommendations


In [8]:
top_recommendations = (
    collab_scores
    .sort_values(["userId", "aggregatedSubmissionScore", "bestSubmissionScore"], ascending=[True, False, False])
    .groupby("userId")
    .head(10)
    .copy()
)

top_recommendations["rank"] = top_recommendations.groupby("userId").cumcount() + 1
top_recommendations = top_recommendations[[
    "userId",
    "rank",
    "projectId",
    "collaborationId",
    "name",
    "status",
    "tagsKey",
    "aggregatedSubmissionScore",
    "bestSubmissionScore",
    "highlightedSubmission",
    "highlightedSubmissionScore",
]]
top_recommendations.head(20)

,userId,rank,projectId,collaborationId,name,status,tagsKey,aggregatedSubmissionScore,bestSubmissionScore,highlightedSubmission,highlightedSubmissionScore
0,user-0002,1,project-001,collab-001-01,Collaboration 1-1,voting,"[acid, breakbeat, cinematic]",0.0,0.0,collaborations/collab-001-01/submissions/sub-0...,0.0
1,user-0002,2,project-001,collab-001-02,Collaboration 1-2,voting,"[breakbeat, lofi, organic]",0.0,0.0,collaborations/collab-001-02/submissions/sub-0...,0.0
2,user-0002,3,project-002,collab-002-01,Collaboration 2-1,submission,"[ambient, drum-and-bass]",0.0,0.0,collaborations/collab-002-01/submissions/sub-0...,0.0
3,user-0002,4,project-002,collab-002-02,Collaboration 2-2,voting,"[idm, uk-bass]",0.0,0.0,collaborations/collab-002-02/submissions/sub-0...,0.0
4,user-0002,5,project-002,collab-002-03,Collaboration 2-3,submission,[minimal],0.0,0.0,collaborations/collab-002-03/submissions/sub-0...,0.0
5,user-0002,6,project-003,collab-003-01,Collaboration 3-1,submission,"[breakbeat, dub, jungle]",0.0,0.0,collaborations/collab-003-01/submissions/sub-0...,0.0
6,user-0002,7,project-003,collab-003-02,Collaboration 3-2,submission,[cinematic],0.0,0.0,collaborations/collab-003-02/submissions/sub-0...,0.0
7,user-0002,8,project-003,collab-003-03,Collaboration 3-3,submission,"[cinematic, lofi, minimal]",0.0,0.0,collaborations/collab-003-03/submissions/sub-0...,0.0
8,user-0002,9,project-004,collab-004-01,Collaboration 4-1,voting,"[ambient, lofi, minimal]",0.0,0.0,collaborations/collab-004-01/submissions/sub-0...,0.0
9,user-0002,10,project-004,collab-004-02,Collaboration 4-2,submission,"[drum-and-bass, electro, synthwave]",0.0,0.0,collaborations/collab-004-02/submissions/sub-0...,0.0


## Sanity evaluation


In [9]:
history_tags = train_collab_history.groupby("userId")["tagsKey"].apply(lambda rows: set(t for arr in rows for t in arr)).to_dict()
history_projects = train_collab_history.groupby("userId")["projectId"].apply(lambda s: set(s)).to_dict()

sanity_eval = top_recommendations.copy()
sanity_eval["tagOverlap"] = [len(set(tags) & history_tags.get(uid, set())) for uid, tags in zip(sanity_eval["userId"], sanity_eval["tagsKey"])]
sanity_eval["sameSeenProject"] = [pid in history_projects.get(uid, set()) for uid, pid in zip(sanity_eval["userId"], sanity_eval["projectId"])]

sanity_summary = {
    "rows": int(len(sanity_eval)),
    "users": int(sanity_eval["userId"].nunique()),
    "unique_recommended_collabs": int(sanity_eval["collaborationId"].nunique()),
    "catalog_coverage_pct": round(100 * sanity_eval["collaborationId"].nunique() / len(collabs), 2),
    "active_only": bool(sanity_eval["status"].isin(["submission", "voting"]).all()),
    "top10_tag_overlap_pct": round(100 * (sanity_eval["tagOverlap"] > 0).mean(), 2),
    "top10_same_project_seen_pct": round(100 * sanity_eval["sameSeenProject"].mean(), 2),
    "top1_tag_overlap_pct": round(100 * (sanity_eval[sanity_eval["rank"] == 1]["tagOverlap"] > 0).mean(), 2),
    "top1_same_project_seen_pct": round(100 * sanity_eval[sanity_eval["rank"] == 1]["sameSeenProject"].mean(), 2),
}

sanity_summary

{'rows': 1850,
 'users': 185,
 'unique_recommended_collabs': 10,
 'catalog_coverage_pct': 21.28,
 'active_only': True,
 'top10_tag_overlap_pct': np.float64(80.16),
 'top10_same_project_seen_pct': np.float64(40.49),
 'top1_tag_overlap_pct': np.float64(92.43),
 'top1_same_project_seen_pct': np.float64(34.59)}

In [10]:
top_recommendations[top_recommendations["rank"] == 1]["collaborationId"].value_counts().head(10)

collaborationId
collab-001-01    185
Name: count, dtype: int64

## Holdout evaluation


In [11]:
holdout_eval = holdout[["userId", "collaborationId"]].rename(columns={"collaborationId": "heldOutCollaborationId"})
hits = top_recommendations.merge(holdout_eval, on="userId", how="inner")
hits["isHit"] = hits["collaborationId"] == hits["heldOutCollaborationId"]

user_hits = hits.groupby("userId").agg(
    hitAt10=("isHit", "max"),
    reciprocalRank=("rank", lambda s: 1 / s[hits.loc[s.index, "isHit"]].iloc[0] if hits.loc[s.index, "isHit"].any() else 0),
).reset_index()

holdout_summary = {
    "users_evaluated": int(len(user_hits)),
    "hit_rate_at_10": round(float(user_hits["hitAt10"].mean()), 4),
    "mrr_at_10": round(float(user_hits["reciprocalRank"].mean()), 4),
}

holdout_summary

{'users_evaluated': 185, 'hit_rate_at_10': 0.2216, 'mrr_at_10': 0.0558}